# Problema Inverso Transiente

Este *notebook* tem como objetivo implementar uma PINN para resolver um problema inverso transisente, ou seja, que evolui no tempo.

**Autor**: Edélio Gabriel Magalhães de Jesus.

---

> ATENÇÃO: Esse *notebbok* será melhor aproveitado de for lido após o *notebbok* `04_inverse_stationary.ipynb`, onde foi explicado o que é um problema inverso e implementada uma PINN a um problema inverso estacionário 1D.
>
> Aqui, a diferença será apenas a inclusão de uma condição inicial, o que implica na evolução temporal do problema.

---

## O nosso problema

Como discutido anteriormente, problemas inversos buscam determinar causas desconhecidas a partir de efeitos observados. Nesse exemplo, trabalharemos com a **equação de difusão 2D transiente** — descrevendo o espalhamento de uma espécie química em um meio homogêneo. Ele foi inspirado na implementação discutida por Tao et. al. [[ref]](#artigo-base). No artugo original, os autores preveem o parâmetro D enquanto uma função, aqui, simplificamos para D constante, o que significa um meio homogêneo.

---
### ``Condições de contorno e iniciais``

Nosso domínio espacial é um quadrado simples, e nosso tempo evolui de 0 a 1:

$$(x,y) \in [0,1]^2, \quad t \in [0,1] \tag{1}$$

A condição inicial é um **pacote gaussiano** centrado em $(0.5, 0.5)$:

$$
c(x, y, 0) = \exp\left(-\frac{(x-0.5)^2 + (y-0.5)^2}{2\sigma^2}\right) \tag{1}
$$

representando uma concentração localizada no centro do domínio no instante inicial. As condições de contorno são de Dirichlet homogêneas:

$$
c = 0 \quad \text{em toda a fronteira } \partial\Omega \tag{1}
$$

À medida que o tempo avança, o pacote gaussiano se alarga e sua amplitude diminui — a concentração se redistribui uniformemente até se dissipar nas bordas.

Por fim, adicionamos um ruído gaussiano aos valores da solução numérica. Aqui, vale destacar a diferença entre:

- Solução numérica: usada para a validação;
- Dados sintéticos (solução numérica mais ruído gaussiano): faz parte do treino - enquanto dados observados.
  
---
### `Requisitos teóricos`

#### **Contexto físico**

Quando uma concentração localizada de uma espécie química é introduzida em um meio — por exemplo, uma proteína em solução aquosa — ela se espalha espontaneamente ao longo do tempo devido ao movimento térmico das moléculas. Esse fenômeno é chamado de **difusão** e é governado pela segunda lei de Fick.

O coeficiente de difusão $D$ quantifica a velocidade desse espalhamento:

- materiais com $D$ alto se difundem rapidamente;
- materiais com $D$ baixo se difundem lentamente.

Conhecer $D$ é fundamental em diversas aplicações:

- transporte de fármacos em tecidos biológicos;
- caracterização de biomoléculas em solução;
- processos de separação e filtração.

<div style="text-align: center;">
  <img 
    src="https://upload.wikimedia.org/wikipedia/commons/4/4d/DiffusionMicroMacro.gif"
    alt="Representação Molecular de difusão"
    style="max-width: 400px; width: 50%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://pt.wikipedia.org/wiki/Difus%C3%A3o_molecular' target='_blank'>
    Wikipedia — Difusão Molecular
  </a>
</div>

#### **A equação de difusão 2D**

A evolução temporal do campo de concentração $c(x, y, t)$ é descrita pela equação de difusão:

$$
\frac{\partial c}{\partial t} = D\left(\frac{\partial^2 c}{\partial x^2} + \frac{\partial^2 c}{\partial y^2}\right) \tag{4}
$$

onde $D$ é o coeficiente de difusão — assumido constante e homogêneo neste exemplo - e c é a concentração da espécie química.

#### **O problema inverso**

Em um experimento real, medir $D$ diretamente não é trivial. O que se observa são perfis de concentração $c(x, y, t)$ em alguns instantes — por exemplo, via microscopia de fluorescência ou imagens de concentração.

O problema inverso que propomos é: **a partir de medições esparsas e ruidosas de $c(x, y, t)$ no interior do domínio espaço-temporal, recuperar $D$**.

Matematicamente:

- problema direto:

$$D \longrightarrow c(x, y, t)$$

- problema inverso:

$$c(x, y, t) \longrightarrow D$$

Esse cenário aparece diretamente em aplicações como:

- caracterização de coeficientes de difusão de proteínas em solução;
- transporte de fármacos em tecidos heterogêneos;
- análise de processos de mistura em microfluídica.

Nesse exemplo, utilizaremos medições sintéticas ruidosas geradas a partir da solução numérica para estimar o valor desconhecido de $D$.

> **Diferença em relação ao exemplo anterior:** no problema de Poisson-Boltzmann, o parâmetro desconhecido era uma **condição de contorno** ($\tilde{\psi}_0$). Aqui, $D$ aparece diretamente na **EDP** — o que torna o gradiente que chega até $D$ durante o treinamento mais indireto, passando pelas derivadas espaciais da rede.

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex05_pinn_inverse_transient.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "notebook"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_transient_2d
from ex05_pinn_inverse_transient import (
    generate_synthetic_data_diffusion,
    train_diffusion,
    evaluate_diffusion
)
from plot_utils import plot_loss, plot_D_evolution, plot_diffusion_snapshots


### **Parâmetros do problema**

Os parâmetros da arquitetura MLP e da amostragem foram escolhidos levando em consideração o exemplo estacionário do *notebook* `04_inverse_stationary.ipynb`, aumentando a largura devido ao avanço temporal.

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS   = 3      
N_OUTPUTS  = 1
N_HIDDEN   = 32
N_LAYERS   = 4
ACTIVATION = nn.Tanh

# Parâmetros do problema
D_TRUE   = 0.01
SIGMA    = 0.1

# Parâmetros de amostragem
N_COLLOC = 5000
N_IC     = 500
N_BC     = 200
N_OBS    = 200
NOISE    = 0.01

# Parâmetros para o treinamento
N_EPOCHS = 10000
LR       = 1e-3
W_PDE      = 1.0
W_BC       = 1.0
W_DATA     = 1.0

Usando: cpu


### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION).to(DEVICE)
D     = nn.Parameter(torch.tensor([0.5], dtype=torch.float32, device=DEVICE))

> ### OBSERVE QUE...

...aqui surge a principal diferença entre uma PINN para um problema direto e uma PINN para um problema inverso.

No problema direto, todos os parâmetros físicos da equação são conhecidos, e a rede neural aprende apenas a função solução $\psi(x)$.

Já no problema inverso, parte da física é desconhecida. Nesse caso, além dos pesos e vieses da rede neural, também queremos estimar um parâmetro físico do sistema — aqui, o potencial de superfície $\psi_0$.

Isso é feito através da instrução:

```python
psi0 = nn.Parameter(torch.tensor([1.0], dtype=torch.float32, device=DEVICE))

### **Amostragem dos pontos**

In [5]:
# ── Amostragem ─────────────────────────────────────────────────────────────────

IC_FN = lambda x, y: torch.exp(
    -((x - 0.5)**2 + (y - 0.5)**2) / (2 * SIGMA**2)
)

X_COLLOC = sample_collocation_rectangular(
    N_COLLOC,
    [0,0,0],
    [1,1,1],
    DEVICE
)

X_IC, C_IC, X_BC, C_BC = sample_boundary_rectangular_transient_2d(
    N_IC,
    N_BC,
    0, 1,
    0, 1,
    0, 1,
    IC_FN,
    DEVICE
)

X_OBS, C_OBS = generate_synthetic_data_diffusion(
    D_true=D_TRUE,
    N_obs=N_OBS,
    noise_amp=NOISE,
    device=DEVICE,
    sigma=SIGMA
)

### **Instanciando o otimizador**

In [6]:
optimizer = torch.optim.Adam(
    list(model.parameters()) + [D],
    lr=LR
)

### **Treinamento**

In [7]:
history = train_diffusion(
    model, D, optimizer,
    X_COLLOC, X_IC, C_IC, X_BC, C_BC, X_OBS, C_OBS,
    N_EPOCHS
)

Epoch 00000 | Loss: 9.54e-02 | Loss PDE: 1.98e-03 | Loss IC: 5.14e-02 | Loss BC: 5.25e-03 | Loss data: 3.68e-02 | D: 0.4990
Epoch 00100 | Loss: 4.86e-02 | Loss PDE: 5.34e-05 | Loss IC: 2.90e-02 | Loss BC: 1.59e-03 | Loss data: 1.80e-02 | D: 0.4883
Epoch 00200 | Loss: 4.80e-02 | Loss PDE: 2.34e-04 | Loss IC: 2.85e-02 | Loss BC: 1.38e-03 | Loss data: 1.79e-02 | D: 0.4108
Epoch 00300 | Loss: 4.66e-02 | Loss PDE: 3.49e-04 | Loss IC: 2.78e-02 | Loss BC: 1.27e-03 | Loss data: 1.72e-02 | D: 0.2388
Epoch 00400 | Loss: 3.61e-02 | Loss PDE: 3.60e-04 | Loss IC: 2.28e-02 | Loss BC: 1.43e-03 | Loss data: 1.16e-02 | D: 0.0121
Epoch 00500 | Loss: 2.46e-02 | Loss PDE: 2.33e-04 | Loss IC: 1.64e-02 | Loss BC: 1.09e-03 | Loss data: 6.95e-03 | D: 0.0012
Epoch 00600 | Loss: 1.52e-02 | Loss PDE: 3.05e-04 | Loss IC: 1.02e-02 | Loss BC: 9.62e-04 | Loss data: 3.65e-03 | D: 0.0014
Epoch 00700 | Loss: 5.24e-03 | Loss PDE: 2.53e-04 | Loss IC: 3.41e-03 | Loss BC: 3.48e-04 | Loss data: 1.23e-03 | D: 0.0006
Epoch 00

### **Visualizando os resultados do treinamento**

Nossa função de perda ficou da seguinte forma:

$$
Loss = \omega_{PDE} * L_{PDE} + \omega_{BC} * L_{BC} + \omega_{CI} * L_{CI} + \omega_{DATA} *  L_{DATA}
$$

Ficou muito semelhante à perda do caso estacionário - devido à mesma justificativa -, com a adição de uma parcela referente à condiçaõ inicial.

No código temos:

---
```python
[...]
    # perda física
    residual = pde_residual_diffusion_2d(model, D, X_col)
    loss_pde = torch.mean(residual ** 2)

    # perda CI
    C_ic_pred = model(X_ic)
    loss_ic   = torch.mean((C_ic_pred - C_ic) ** 2)

    # perda CC
    C_bc_pred = model(X_bc)
    loss_bc   = torch.mean((C_bc_pred - C_bc) ** 2)

    # perda dados
    C_obs_pred = model(X_obs)
    loss_data  = torch.mean((C_obs_pred - C_obs) ** 2)

    loss = w_pde * loss_pde + w_ic * loss_ic + \
           w_bc * loss_bc + w_data * loss_data

    return loss, loss_pde, loss_ic, loss_bc, loss_data
```
---
    

In [11]:
plot_loss(history)

As condições de contorno foram muito bem aprendidas, o que faz sentido, dado que eram homogêneas de Dirichlet. As condições iniciais também chegaram a ordens de grandezas extremamente baixas: $10^{-5}$. 

Como um todo, o modelo resultante apredendeu muito bem nosso problema transiente, tendo como perda total valores da ordem $10^{-4}$

### **Validando o modelo**

As derivadas espaciais de segunda ordem da equação (4) são aproximadas por diferenças centrais. A evolução temporal é então calculada explicitamente a cada passo de tempo:

$$
c^{n+1}
=
c^n
+
D\Delta t
\left(
\frac{\partial^2 c}{\partial x^2}
+
\frac{\partial^2 c}{\partial y^2}
\right) \tag{5}
$$

O passo temporal $\Delta t$ é escolhido com base na condição de estabilidade CFL (Courant–Friedrichs–Lewy), que limita o tamanho do passo de tempo para evitar instabilidades numéricas durante a simulação.

In [16]:
results = evaluate_diffusion(
    model=model,
    D=D,
    D_true=D_TRUE,
    device=DEVICE,
    sigma=SIGMA
)

print(f"D verdadeiro:    {D_TRUE:.4f}")
print(f"D recuperado:    {results['D_pred']:.4f}")
print(f"Erro percentual: {results['error_pct']:.2f}%")
print(f"Erro L2:         {results['l2_error']:.2e}")

D verdadeiro:    0.0100
D recuperado:    0.0100
Erro percentual: 0.12%
Erro L2:         1.22e-02


In [14]:
plot_D_evolution(history, D_TRUE)


In [15]:
plot_diffusion_snapshots(results)

O modelo fez uma previsão praticamente perfeita! Com erro percentual de $0.12\%$.

## Referências

<a id='artigo-base'></a> WANG, Tao et al. Physics-Informed Neural Networks for Solving Forward and Inverse Problems Governed by Partial Differential Equations. arXiv preprint arXiv:2403.03970, 2024. Disponível em: https://arxiv.org/html/2403.03970v1